# MNIST Digit Classification using a Simple ANN

**Author:** Mukilan S  
**Dataset:** MNIST (Modified National Institute of Standards and Technology)  
**Framework:** TensorFlow 2.20 / Keras  

This notebook provides a complete, step-by-step walkthrough of the full machine learning pipeline for classifying handwritten digits (0–9).

---

## Pipeline Overview
1. Load and Explore the Dataset
2. Preprocess & Normalize Data
3. Build an ANN with Dropout Regularization
4. Train with Early Stopping
5. Evaluate: Confusion Matrix & Per-Class Accuracy
6. Compare Multiple Architectures
7. Benchmark Against a CNN
8. Hyperparameter Sensitivity Analysis

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from tensorflow import keras
from tensorflow.keras import layers

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Setup complete.')

---
## 1. Load & Explore the Dataset

MNIST contains **70,000 grayscale images** of handwritten digits (0–9):  
- 60,000 training images  
- 10,000 test images  
- Each image: **28 × 28 pixels**, values 0 (black) – 255 (white)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print(f'Training set : {x_train.shape}  Labels: {y_train.shape}')
print(f'Test set     : {x_test.shape}   Labels: {y_test.shape}')
print(f'Pixel range  : {x_train.min()} – {x_train.max()}')
print(f'Classes      : {np.unique(y_train)}')

In [ ]:
# Visualise one sample per digit
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Images — One Per Digit Class', fontsize=13, fontweight='bold')
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    ax  = axes[digit // 5][digit % 5]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(f'Digit: {digit}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 4))
unique, counts = np.unique(y_train, return_counts=True)
bars = ax.bar(unique, counts, color=plt.cm.tab10(unique/10))
ax.set_title('Class Distribution in Training Set', fontweight='bold')
ax.set_xlabel('Digit'); ax.set_ylabel('Count')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+50,
            str(count), ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print('Dataset is well-balanced — no class imbalance issue.')

---
## 2. Preprocess & Normalize

For a Dense ANN, we need to:
1. **Flatten** the 28×28 image into a 784-element vector  
2. **Normalize** pixel values from `[0, 255]` → `[0.0, 1.0]`  
3. **One-hot encode** labels (e.g., `5` → `[0,0,0,0,0,1,0,0,0,0]`)

In [ ]:
NUM_CLASSES = 10

# Flatten
x_train_flat = x_train.reshape(60000, -1).astype('float32')
x_test_flat  = x_test.reshape(10000, -1).astype('float32')

# Normalize
x_train_flat /= 255.0
x_test_flat  /= 255.0

# One-hot encode
y_train_ohe = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_ohe  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print(f'x_train_flat : {x_train_flat.shape}  range: [{x_train_flat.min():.1f}, {x_train_flat.max():.1f}]')
print(f'x_test_flat  : {x_test_flat.shape}')
print(f'y_train_ohe  : {y_train_ohe.shape}  (one-hot)')
print(f'\nExample label: {y_train[0]} → {y_train_ohe[0]}')

---
## 3. Build the Model

### Architecture
```
Input (784) → Dense(128, ReLU) → Dropout(0.2) → Dense(64, ReLU) → Dropout(0.2) → Dense(10, Softmax)
```

**Why Dropout?**  
Dropout randomly deactivates 20% of neurons during each training step, forcing the network to learn redundant representations and preventing overfitting.

In [ ]:
def build_ann(units=(128, 64), dropout_rate=0.2):
    model_layers = [keras.Input(shape=(784,))]
    for u in units:
        model_layers.append(layers.Dense(u, activation='relu'))
        model_layers.append(layers.Dropout(dropout_rate))
    model_layers.append(layers.Dense(10, activation='softmax', name='output'))
    return keras.Sequential(model_layers)

model = build_ann(units=(128, 64), dropout_rate=0.2)
model.summary()

---
## 4. Train with Early Stopping

**Early Stopping** monitors validation loss and stops training when it stops improving (with patience=3 epochs), restoring the best weights automatically.

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    x_train_flat, y_train_ohe,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training History — ANN (128→64) with Dropout', fontweight='bold')

ax1.plot(history.history['accuracy'],     label='Train', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val',   lw=2, ls='--')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train', lw=2)
ax2.plot(history.history['val_loss'], label='Val',   lw=2, ls='--')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 5. Evaluation
### 5a. Overall Accuracy

In [ ]:
test_loss, test_acc = model.evaluate(x_test_flat, y_test_ohe, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc*100:.2f}%')

### 5b. Confusion Matrix

The confusion matrix shows which digits are confused with each other. Darker = more predictions. The diagonal is correct classifications.

In [ ]:
y_pred = np.argmax(model.predict(x_test_flat, verbose=0), axis=1)
cm     = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — MNIST ANN', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 5c. Per-Class Accuracy (Precision / Recall / F1)

In [ ]:
report = classification_report(
    y_test, y_pred,
    target_names=[f'Digit {i}' for i in range(10)]
)
print(report)

In [ ]:
# Visualise per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(10), per_class_acc,
              color=[plt.cm.RdYlGn(a/100) for a in per_class_acc])
ax.axhline(y=per_class_acc.mean(), color='navy', ls='--', lw=1.5,
           label=f'Mean: {per_class_acc.mean():.1f}%')
ax.set_xticks(range(10))
ax.set_xticklabels([f'Digit {i}' for i in range(10)])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1, f'{acc:.1f}%',
            ha='center', fontsize=8)
plt.tight_layout(); plt.show()

### 5d. Sample Predictions

In [ ]:
preds = np.argmax(model.predict(x_test_flat[:10], verbose=0), axis=1)

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
fig.suptitle('Sample Predictions (green = correct, red = wrong)',
             fontsize=12, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i], cmap='gray')
    color = 'green' if preds[i] == y_test[i] else 'red'
    ax.set_title(f'Pred:{preds[i]}  True:{y_test[i]}',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 6. Architecture Comparison

We train three ANN architectures to understand the tradeoff between model complexity and accuracy.

In [ ]:
architectures = {
    'Small (64→32)':     (64, 32),
    'Medium (128→64)':   (128, 64),
    'Large (256→128→64)':(256, 128, 64),
}

arch_results = {}
for name, units in architectures.items():
    m = build_ann(units=units)
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m.fit(x_train_flat, y_train_ohe, epochs=15, batch_size=64,
          validation_split=0.1, callbacks=[es], verbose=0)
    _, acc = m.evaluate(x_test_flat, y_test_ohe, verbose=0)
    arch_results[name] = {'accuracy': acc*100, 'params': m.count_params()}
    print(f'{name:25s}  Acc: {acc*100:.2f}%  Params: {m.count_params():,}')

In [ ]:
names  = list(arch_results.keys())
accs   = [arch_results[n]['accuracy'] for n in names]
params = [arch_results[n]['params']   for n in names]
colors = ['#3b82f6', '#10b981', '#f59e0b']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ANN Architecture Comparison', fontweight='bold')

bars = ax1.bar(names, accs, color=colors, edgecolor='white', lw=1.5)
ax1.set_ylim(95, 100); ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Accuracy')
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{acc:.2f}%', ha='center', fontweight='bold')
ax1.tick_params(axis='x', rotation=15); ax1.grid(axis='y', alpha=0.3)

bars2 = ax2.bar(names, params, color=colors, edgecolor='white', lw=1.5)
ax2.set_ylabel('Parameters'); ax2.set_title('Model Complexity')
for bar, p in zip(bars2, params):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
             f'{p:,}', ha='center', fontsize=9)
ax2.tick_params(axis='x', rotation=15); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

---
## 7. CNN Benchmark

Convolutional Neural Networks are better suited for image data because they exploit spatial relationships between pixels using learnable filters. Here we compare our best ANN to a simple CNN.

In [ ]:
# Reshape for CNN: (N, 28, 28, 1)
x_train_cnn = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test_cnn  = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax'),
])
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
cnn.fit(x_train_cnn, y_train_ohe, epochs=15, batch_size=64,
        validation_split=0.1, callbacks=[es], verbose=1)

In [ ]:
_, cnn_acc = cnn.evaluate(x_test_cnn, y_test_ohe, verbose=0)
best_ann   = max(accs)

print(f'Best ANN Accuracy : {best_ann:.2f}%  ({max(arch_results, key=lambda k: arch_results[k]["accuracy"])})')
print(f'CNN  Accuracy     : {cnn_acc*100:.2f}%')
print(f'CNN  Parameters   : {cnn.count_params():,}')
print(f'\nCNN gains {cnn_acc*100-best_ann:.2f}% accuracy at the cost of {(cnn.count_params()-max(params)):,} extra parameters.')

---
## 8. Hyperparameter Sensitivity Analysis

Understanding how hyperparameters affect performance is key to being a good ML engineer.

In [ ]:
# Batch size analysis
batch_sizes = [16, 32, 64, 128, 256]
batch_accs  = []
for bs in batch_sizes:
    m = build_ann(); m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m.fit(x_train_flat, y_train_ohe, epochs=10, batch_size=bs,
          validation_split=0.1, callbacks=[es], verbose=0)
    _, acc = m.evaluate(x_test_flat, y_test_ohe, verbose=0)
    batch_accs.append(acc*100)
    print(f'Batch {bs:3d}: {acc*100:.2f}%')

In [ ]:
# Dropout rate analysis
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
dropout_accs  = []
for dr in dropout_rates:
    m = build_ann(dropout_rate=dr); m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m.fit(x_train_flat, y_train_ohe, epochs=10,
          validation_split=0.1, callbacks=[es], verbose=0)
    _, acc = m.evaluate(x_test_flat, y_test_ohe, verbose=0)
    dropout_accs.append(acc*100)
    print(f'Dropout {dr}: {acc*100:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Hyperparameter Sensitivity Analysis', fontweight='bold')

ax1.plot([str(b) for b in batch_sizes], batch_accs, 'o-', color='#3b82f6', lw=2, ms=8)
ax1.set_title('Batch Size vs Accuracy')
ax1.set_xlabel('Batch Size'); ax1.set_ylabel('Test Accuracy (%)')
ax1.grid(True, alpha=0.3)

ax2.plot([str(d) for d in dropout_rates], dropout_accs, 's-', color='#10b981', lw=2, ms=8)
ax2.set_title('Dropout Rate vs Accuracy')
ax2.set_xlabel('Dropout Rate'); ax2.set_ylabel('Test Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Optimal batch size  : {batch_sizes[np.argmax(batch_accs)]}')
print(f'Optimal dropout rate: {dropout_rates[np.argmax(dropout_accs)]}')

---
## Summary

| Model | Accuracy | Parameters |
|---|---|---|
| ANN Small (64→32) | — | ~52,000 |
| ANN Medium (128→64) | — | ~109,000 |
| ANN Large (256→128→64) | — | ~236,000 |
| CNN (Conv→Conv→Dense) | — | ~94,000 |

**Key Takeaways:**
- The MNIST dataset is well-suited for simple ANNs — even the small model exceeds 97% accuracy.
- CNNs outperform ANNs on image tasks because they exploit spatial structure.
- Dropout regularization consistently prevents overfitting with minimal accuracy loss.
- Batch size affects both speed and generalization — larger batches train faster but may generalize slightly worse.